## This notebook demonstrates access to NOAA Coastal Ocean ReAnalysis (CORA) data that is stored in an S3 bucket and made freely available via NOAA's Open Data Dissemination (NODD) service. The data will be loaded as an xarray dataset. For a use case example, the maximum water level value at all CORA GEC nodes for an input time slice is extracted and plottted.

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Signell, R. (2024). Reproducible Notebooks/hurricane-ike-water-levels. https://github.com/reproducible-notebooks/hurricane-ike-water-levels/tree/master***

_ _ 

### Input a time slice of interest

In [ ]:
time_slice = slice("1989-09-21", "1989-09-23")  # Hurricane Hugo

### Import packages ###

In [ ]:
import cartopy.crs as ccrs
import cmocean
import geoviews as gv
from holoviews.operation.datashader import rasterize
import intake
import numpy as np
import pandas as pd
import zarr

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-GEC water levels (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_gec/native_grid/water_levels/)). ###
#### This dataset contains hourly water levels from 1979-2022 in the GEC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **map** chunking here, since we are interested in a small time slice at many points.

In [ ]:
ds = catalog["CORA-GEC-fort.63-map"].to_dask()
ds

### Compute the maximum water level at every node over the specified time slice.

In [ ]:
wl_max = ds.sel(time=time_slice)['zeta'].max(dim='time',skipna=False).compute()

### Make an interactive map of the values.
#### The color scale changes dynamically as you zoom in/out.

In [ ]:
gv.extension('bokeh')

verts = pd.DataFrame({
    'x': ds.x.values, 
    'y': ds.y.values, 
    'var': wl_max.values
})
tris = pd.DataFrame(ds.sel(time=time_slice)['element'].values.astype(int) - 1, columns=['v0', 'v1', 'v2'])

points = gv.Points(verts, vdims=['var'], crs=ccrs.PlateCarree())
trimesh = gv.TriMesh((tris, points), crs=ccrs.PlateCarree(), label='Maximum water level (mm, MSL)')

mesh = rasterize(trimesh).opts(
    cmap=cmocean.cm.balance, 
    colorbar=True, 
    width=1000, 
    height=700
)

gv.tile_sources.OSM * mesh